# CMAPSS 전처리 — 실험조건 기반 CSV 생성 + MLflow 기록

In [1]:
# ============================================================
# [0] 실험조건 설정
# ============================================================
DATASET_ID   = 'FD001'
RUL_CAP      = 125
GAUSS_SIGMA  = 2.0
RANDOM_STATE = 42

# 파생변수 ON/OFF + 개별 파라미터
USE_MA   = True;  WIN_MA   = 5    # Rolling Mean window
USE_DIFF = True;  PER_DIFF = 1    # Diff periods (몇 스텝 차분)
USE_LAG  = True;  LAG_STEP = 1    # Lag step (몇 스텝 전 값)
USE_STD  = True;  WIN_STD  = 5    # Rolling Std window
USE_EMA  = True;  SPAN_EMA = 5    # EMA span

# 정규화 방식: 'minmax' | 'standard' | 'robust'
SCALER_TYPE = 'minmax'

# 센서 
CONST_COLS_2 = [] # 추가로 빼고싶은 센서 (예 : CONST_COLS_2 = ['s_9', 's_14'])

# 파일명 태그 — 켜진 파생변수와 파라미터를 모두 반영
tag_parts = []
if USE_MA:   tag_parts.append(f'MA{WIN_MA}')
if USE_DIFF: tag_parts.append(f'D{PER_DIFF}')
if USE_LAG:  tag_parts.append(f'L{LAG_STEP}')
if USE_STD:  tag_parts.append(f'S{WIN_STD}')
if USE_EMA:  tag_parts.append(f'E{SPAN_EMA}')
feat_flags = '-'.join(tag_parts) if tag_parts else 'none'

EXP_TAG = f"cap{RUL_CAP}_sig{int(GAUSS_SIGMA)}_{feat_flags}_{SCALER_TYPE}"
OUT_DIR = 'preprocessed'

import os
os.makedirs(OUT_DIR, exist_ok=True)
print(f"실험 태그: {DATASET_ID}_{EXP_TAG}")
print(f"저장 경로: {OUT_DIR}/")

실험 태그: FD001_cap125_sig2_MA5-D1-L1-S5-E5_minmax
저장 경로: preprocessed/


In [2]:
# ============================================================
# [1] 환경 설정 및 라이브러리 로드
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
from scipy.ndimage import gaussian_filter1d
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.feature_selection import mutual_info_regression
from scipy.stats import ttest_ind
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import mlflow

# 한글 폰트 설정 (Windows 기준)
matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

# 데이터 경로 설정 (사용자 환경에 맞게 수정)
DATA_PATH = './CMAPSSData/'

# MLflow 실험 설정
mlflow.set_experiment(f'CMAPSS_{DATASET_ID}')

# 스케일러 매핑
SCALERS = {'minmax': MinMaxScaler, 'standard': StandardScaler, 'robust': RobustScaler}

print("✅ 환경 설정 및 라이브러리 로드 완료")

✅ 환경 설정 및 라이브러리 로드 완료


In [3]:
# ============================================================
# [2] 데이터 로드
# ============================================================
def load_cmapss_data(data_id='FD001'):
    cols = ['unit_nr', 'time_cycles', 'setting_1', 'setting_2', 'setting_3'] + [f's_{i}' for i in range(1, 22)]
    train = pd.read_csv(f'{DATA_PATH}train_{data_id}.txt', sep=r'\s+', header=None, names=cols)
    test  = pd.read_csv(f'{DATA_PATH}test_{data_id}.txt',  sep=r'\s+', header=None, names=cols)
    rul   = pd.read_csv(f'{DATA_PATH}RUL_{data_id}.txt',   sep=r'\s+', header=None, names=['RUL'])
    return train, test, rul

df_train, df_test, df_rul = load_cmapss_data(DATASET_ID)
print(f" {DATASET_ID} 로드 완료: Train {df_train.shape}, Test {df_test.shape}")

 FD001 로드 완료: Train (20631, 26), Test (13096, 26)


In [4]:
# ============================================================
# [3] 전처리 - 상수 제거 / RUL Capping / Gaussian 스무딩
# ============================================================
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white", font='Malgun Gothic')

# 단계별 데이터 보관용 리스트 및 센서 설정

# [Step 2] 상수 제거 & RUL Capping
df_train['max_cycle'] = df_train.groupby('unit_nr')['time_cycles'].transform('max')
df_train['RUL'] = (df_train['max_cycle'] - df_train['time_cycles']).clip(upper=RUL_CAP)
df_train.drop(columns=['max_cycle'], inplace=True)

# 상수 센서 자동 감지 (std == 0)
sensor_cols = [c for c in df_train.columns if c.startswith('s_')]
STD_EPSILON = 1e-2
auto_const = [c for c in sensor_cols if df_train[c].std() < STD_EPSILON]
CONST_COLS  = sorted(set(auto_const + CONST_COLS_2))
print(f"자동 감지된 상수 센서: {auto_const}")
print(f"자동감지+추가제거센서 : {CONST_COLS}")
print(f"자동 감지된 사실상 상수 센서 ({len(auto_const)}개, std<{STD_EPSILON}): {auto_const}")

df_train = df_train.drop(columns=CONST_COLS)
df_test  = df_test.drop(columns=CONST_COLS)

base_fe = sorted([c for c in df_train.columns if c.startswith('s_')], 
                 key=lambda x: int(x.split('_')[1]))
print(f"남는 센서: {base_fe}")

# [Step 3] 가우시안 스무딩
active_sensors = [c for c in df_train.columns if c.startswith('s_')]

def apply_gaussian_team_style(df, features, sigma=GAUSS_SIGMA): #################################
    df = df.copy().sort_values(['unit_nr','time_cycles']).reset_index(drop=True)
    df[features] = df[features].astype(np.float32)
    for uid in df['unit_nr'].unique():
        mask = df['unit_nr'] == uid
        for col in features:
            arr = df.loc[mask, col].to_numpy(dtype=np.float32)
            df.loc[mask, col] = gaussian_filter1d(arr, sigma=sigma, mode='nearest')
    return df

df_train = apply_gaussian_team_style(df_train, active_sensors, sigma=GAUSS_SIGMA)
df_test  = apply_gaussian_team_style(df_test,  active_sensors, sigma=GAUSS_SIGMA)

print("✅ 전처리(상수제거/RUL cap/Gaussian) 완료")

자동 감지된 상수 센서: ['s_1', 's_5', 's_6', 's_10', 's_16', 's_18', 's_19']
자동감지+추가제거센서 : ['s_1', 's_10', 's_16', 's_18', 's_19', 's_5', 's_6']
자동 감지된 사실상 상수 센서 (7개, std<0.01): ['s_1', 's_5', 's_6', 's_10', 's_16', 's_18', 's_19']
남는 센서: ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14', 's_15', 's_17', 's_20', 's_21']
✅ 전처리(상수제거/RUL cap/Gaussian) 완료


In [5]:
# ============================================================
# [4] 파생변수 생성 + Train/Val/Test 분리 + 정규화
# ============================================================
# 1. 정렬
df_train = df_train.sort_values(['unit_nr', 'time_cycles']).reset_index(drop=True)
df_test  = df_test.sort_values(['unit_nr', 'time_cycles']).reset_index(drop=True)

# 2. split 먼저
unit_ids = df_train['unit_nr'].unique()
train_units, val_units = train_test_split(unit_ids, test_size=0.2, random_state=RANDOM_STATE)

train_set = df_train[df_train['unit_nr'].isin(train_units)].copy()
val_set   = df_train[df_train['unit_nr'].isin(val_units)].copy()
test_set  = df_test.copy()

# 3. scaling 먼저 (원본 base + setting만)
base_features = base_fe
setting_features = ['setting_1', 'setting_2', 'setting_3']
y_label = 'RUL'

base_plus_setting = base_features + setting_features

# 스위치(SCALER_TYPE)로 스케일러 선택
scaler = SCALERS[SCALER_TYPE]()
train_set[base_plus_setting] = scaler.fit_transform(train_set[base_plus_setting])
val_set[base_plus_setting]   = scaler.transform(val_set[base_plus_setting])
test_set[base_plus_setting]  = scaler.transform(test_set[base_plus_setting])

# 4. 파생변수 생성 — USE_* 스위치 + 개별 파라미터
def add_features(df, features):
    df_res = df.copy()
    for col in features:
        group = df_res.groupby('unit_nr')[col]

        # 1. Rolling Mean (MA)
        if USE_MA:
            df_res[f'{col}_ma'] = group.transform(lambda x: x.rolling(WIN_MA, min_periods=1).mean())

        # 2. Diff (차분, periods 조절 가능)
        if USE_DIFF:
            df_res[f'{col}_diff'] = group.transform(lambda x: x.diff(PER_DIFF).fillna(0))

        # 3. Lag (시차, LAG_STEP 조절 가능)
        if USE_LAG:
            df_res[f'{col}_lag'] = group.transform(lambda x: x.shift(LAG_STEP).bfill())

        # 4. Rolling Std
        if USE_STD:
            df_res[f'{col}_std'] = group.transform(lambda x: x.rolling(WIN_STD, min_periods=1).std().fillna(0))

        # 5. EMA
        if USE_EMA:
            df_res[f'{col}_ema'] = group.transform(lambda x: x.ewm(span=SPAN_EMA, adjust=False).mean())

    return df_res

train_set = add_features(train_set, base_features)
val_set   = add_features(val_set,   base_features)
test_set  = add_features(test_set,  base_features)

# 5. 최종 피처 리스트 (USE_* 스위치 반영)
X_features_full = base_features + setting_features
if USE_MA:   X_features_full += [f'{c}_ma'   for c in base_features]
if USE_DIFF: X_features_full += [f'{c}_diff' for c in base_features]
if USE_LAG:  X_features_full += [f'{c}_lag'  for c in base_features]
if USE_STD:  X_features_full += [f'{c}_std'  for c in base_features]
if USE_EMA:  X_features_full += [f'{c}_ema'  for c in base_features]

# 6. 최종 변수
X_train = train_set[X_features_full]
y_train = train_set[y_label]

X_val = val_set[X_features_full]
y_val = val_set[y_label]

X_test = test_set.reset_index(drop=True).groupby('unit_nr').last()[X_features_full]
y_test = df_rul['RUL'].values

print(f"✅ 파생변수 생성 완료  |  피처 수: {len(X_features_full)}개  |  스케일러: {SCALER_TYPE}")

✅ 파생변수 생성 완료  |  피처 수: 87개  |  스케일러: minmax


In [8]:
# ============================================================
# [5] CSV 저장 + MLflow 기록
# ============================================================

# 다음 노트북(모델 학습)이 요구하는 6개 파일
subtrain_path = f'{OUT_DIR}/{DATASET_ID}_{EXP_TAG}_subtrain_preprocessed.csv'
valid_path    = f'{OUT_DIR}/{DATASET_ID}_{EXP_TAG}_valid_preprocessed.csv'
#test_path     = f'{OUT_DIR}/{DATASET_ID}_{EXP_TAG}_test_preprocessed.csv'
#rul_path      = f'{OUT_DIR}/{DATASET_ID}_{EXP_TAG}_test_RUL.csv'
eval_path     = f'{OUT_DIR}/{DATASET_ID}_{EXP_TAG}_test_eval.csv'
feat_path     = f'{OUT_DIR}/{DATASET_ID}_{EXP_TAG}_feature_columns.txt'

# train/val/test 저장 (unit_nr, time_cycles, RUL, 피처 전부 포함)
train_set.to_csv(subtrain_path, index=False)
val_set.to_csv(valid_path,    index=False)
# test_set.to_csv(test_path,    index=False)

# RUL 파일
# df_rul.to_csv(rul_path, index=False)

# test_eval: 엔진별 마지막 사이클 + 실제 RUL 병합
test_eval = (test_set.sort_values(['unit_nr','time_cycles'])
                     .groupby('unit_nr').last().reset_index())
test_eval['RUL'] = df_rul['RUL'].values
test_eval.to_csv(eval_path, index=False)

# feature_columns.txt (전체 컬럼 나열 → 다음 노트북이 unit_nr/time_cycles/RUL 제외)
with open(feat_path, 'w', encoding='utf-8') as f:
    for c in train_set.columns:
        f.write(c + '\n')

print("✅ CSV 저장 완료")
saved_paths = [subtrain_path, valid_path, eval_path, feat_path]
for p in saved_paths:
    print(f"   {p}")

# ---- MLflow 기록 ----
with mlflow.start_run(run_name=f'preprocess_{EXP_TAG}'):
    mlflow.log_params({
        'dataset_id'   : DATASET_ID,
        'RUL_CAP'      : RUL_CAP,
        'GAUSS_SIGMA'  : GAUSS_SIGMA,
        'USE_MA'       : USE_MA,       'WIN_MA'   : WIN_MA,
        'USE_DIFF'     : USE_DIFF,     'PER_DIFF' : PER_DIFF,
        'USE_LAG'      : USE_LAG,      'LAG_STEP' : LAG_STEP,
        'USE_STD'      : USE_STD,      'WIN_STD'  : WIN_STD,
        'USE_EMA'      : USE_EMA,      'SPAN_EMA' : SPAN_EMA,
        'SCALER_TYPE'  : SCALER_TYPE,
        'random_state' : RANDOM_STATE,
        'n_features'   : len(X_features_full),
        'n_train_units': len(train_units),
        'n_val_units'  : len(val_units),
    })
    mlflow.log_metrics({
        'train_rows': len(train_set),
        'val_rows'  : len(val_set),
        'test_rows' : len(test_set),
    })
    for p in saved_paths:
        mlflow.log_artifact(p, artifact_path='preprocessed')

print(f"\n✅ MLflow 기록 완료")
print(f"   experiment = CMAPSS_{DATASET_ID}")
print(f"   run        = preprocess_{EXP_TAG}")

✅ CSV 저장 완료
   preprocessed/FD001_cap125_sig2_MA5-D1-L1-S5-E5_minmax_subtrain_preprocessed.csv
   preprocessed/FD001_cap125_sig2_MA5-D1-L1-S5-E5_minmax_valid_preprocessed.csv
   preprocessed/FD001_cap125_sig2_MA5-D1-L1-S5-E5_minmax_test_eval.csv
   preprocessed/FD001_cap125_sig2_MA5-D1-L1-S5-E5_minmax_feature_columns.txt

✅ MLflow 기록 완료
   experiment = CMAPSS_FD001
   run        = preprocess_cap125_sig2_MA5-D1-L1-S5-E5_minmax


## 다음 노트북(모델 학습) 연결 방법

모델 학습 노트북의 **Cell 1 (데이터 로드)** 에서 파일명을 `EXP_TAG` 포함하도록 수정:

```python
DATA_DIR   = 'preprocessed'
DATASET_ID = 'FD001'
EXP_TAG    = 'cap125_sig2_MA5-D1-L1-S5-E5_minmax'   # 전처리 노트북과 동일 값

feat_path = os.path.join(DATA_DIR, f'{DATASET_ID}_{EXP_TAG}_feature_columns.txt')
with open(feat_path, 'r', encoding='utf-8') as f:
    all_cols = [l.strip() for l in f if l.strip()]
MODEL_FEATURES = [c for c in all_cols if c not in ['unit_nr','time_cycles','RUL']]

subtrain  = pd.read_csv(os.path.join(DATA_DIR, f'{DATASET_ID}_{EXP_TAG}_subtrain_preprocessed.csv'))
valid     = pd.read_csv(os.path.join(DATA_DIR, f'{DATASET_ID}_{EXP_TAG}_valid_preprocessed.csv'))
test_eval = pd.read_csv(os.path.join(DATA_DIR, f'{DATASET_ID}_{EXP_TAG}_test_eval.csv'))
```

각 모델 학습 루프 안에 MLflow 기록 추가 (RMSE / NASA Score):

```python
with mlflow.start_run(run_name=f'{mname}_{EXP_TAG}'):
    mlflow.log_params(best_p)
    mlflow.log_metrics({
        'cv_rmse'  : cv_rmse,
        'val_rmse' : val_m['RMSE'],  'val_nasa' : val_m['NASA_Score'],
        'test_rmse': test_m['RMSE'], 'test_nasa': test_m['NASA_Score'],
        'test_mae' : test_m['MAE'],  'test_r2'  : test_m['R2'],
    })
    mlflow.sklearn.log_model(best_m, mname)
```
